In [2]:
import pandas as pd
import altair as alt
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

ap_data = pd.read_csv("data/processed/ap_state_data.csv")

In [3]:
# ==================================================
# TASK 3: EXPLORING STATE AP POLICIES
# ==================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


# --------------------------------------------------
# POLICY LABELS
# --------------------------------------------------

policy_labels = {
    "performance_incentive": "Performance Incentive",
    "participation_incentive": "Participation Incentive",
    "teacher_training": "Teacher Training",
    "exam_fee_support": "AP Exam Fee Support",
    "access_support": "AP Access Support",
    "accountability": "AP Accountability / Reporting"
}


# --------------------------------------------------
# WIDGET CONTROLS
# --------------------------------------------------

policy_widget = widgets.Dropdown(
    options=[
        (label, column)
        for column, label in policy_labels.items()
    ],
    value="performance_incentive",
    description="AP Policy:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="390px")
)

state_widget = widgets.Dropdown(
    options=sorted(ap_data["state"].tolist()),
    value="Florida",
    description="Highlight State:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="390px")
)


# --------------------------------------------------
# OUTPUT AREA FOR ALTAIR DASHBOARD
# --------------------------------------------------

task3_output = widgets.Output()


# --------------------------------------------------
# DRAW TASK 3
# --------------------------------------------------

def draw_task3(policy, highlighted_state):

    # ----------------------------------------------
    # Add temporary columns used for visual encoding
    # ----------------------------------------------

    task3_data = ap_data.copy()

    task3_data["policy_status"] = task3_data[policy].map({
        1: "Has selected policy",
        0: "Does not have selected policy"
    })

    task3_data["state_highlight"] = task3_data["state"].apply(
        lambda x: "Highlighted State"
        if x == highlighted_state
        else "Other State"
    )


    # ----------------------------------------------
    # SHARED COLOR ENCODING
    # ----------------------------------------------

    policy_color = alt.Color(
        "policy_status:N",
        scale=alt.Scale(
            domain=[
                "Has selected policy",
                "Does not have selected policy"
            ],
            range=[
                "steelblue",
                "lightgray"
            ]
        ),
        legend=None
    )


    # ----------------------------------------------
    # SCATTERPLOT BASE
    # ----------------------------------------------

    task3_base = alt.Chart(task3_data).encode(
        x=alt.X(
            "participation_2025:Q",
            title="AP Participation Rate",
            axis=alt.Axis(format=".0%")
        ),
        y=alt.Y(
            "pass_rate_2025:Q",
            title="AP Exam Pass Rate",
            axis=alt.Axis(format=".0%")
        )
    )


    # ----------------------------------------------
    # SCATTERPLOT POINTS
    # ----------------------------------------------

    task3_points = task3_base.mark_circle(
        size=200
    ).encode(
        color=policy_color,
        tooltip=[
            "state:N",
            alt.Tooltip(
                "participation_2025:Q",
                title="Participation",
                format=".1%"
            ),
            alt.Tooltip(
                "pass_rate_2025:Q",
                title="Pass Rate",
                format=".1%"
            ),
            alt.Tooltip(
                "policy_status:N",
                title="Selected Policy"
            )
        ]
    )


    # ----------------------------------------------
    # STATE ABBREVIATIONS
    # ----------------------------------------------

    task3_labels = task3_base.mark_text(
        fontSize=8
    ).encode(
        text="state_abbr:N"
    )


    # ----------------------------------------------
    # HIGHLIGHTED STATE RING
    # ----------------------------------------------

    task3_selected_state = (
        task3_base
        .transform_filter(
            alt.datum.state == highlighted_state
        )
        .mark_circle(
            size=350,
            fillOpacity=0,
            stroke="black",
            strokeWidth=3
        )
    )


    task3_scatter = (
        task3_points
        + task3_labels
        + task3_selected_state
    ).properties(
        width=725,
        height=420
    )


    # ----------------------------------------------
    # PARTICIPATION BAR CHART
    # ----------------------------------------------

    task3_participation_bar = (
        alt.Chart(task3_data)
        .mark_bar()
        .encode(
            x=alt.X(
                "participation_2025:Q",
                title="AP Participation Rate",
                scale=alt.Scale(domain=[0, 0.40]),
                axis=alt.Axis(format=".0%")
            ),
            y=alt.Y(
                "state:N",
                title=None,
                sort="-x",
                axis=alt.Axis(
                    labels=False,
                    ticks=False
                )
            ),
            color=policy_color,
            stroke=alt.condition(
                alt.datum.state == highlighted_state,
                alt.value("black"),
                alt.value(None)
            ),
            strokeWidth=alt.condition(
                alt.datum.state == highlighted_state,
                alt.value(2),
                alt.value(0)
            ),
            tooltip=[
                "state:N",
                alt.Tooltip(
                    "participation_2025:Q",
                    title="AP Participation Rate",
                    format=".1%"
                )
            ]
        )
        .properties(
            width=350,
            height=180,
            title="AP Participation Rate"
        )
    )


    # ----------------------------------------------
    # PASS RATE BAR CHART
    # ----------------------------------------------

    task3_pass_rate_bar = (
        alt.Chart(task3_data)
        .mark_bar()
        .encode(
            x=alt.X(
                "pass_rate_2025:Q",
                title="AP Exam Pass Rate",
                scale=alt.Scale(domain=[0, 0.90]),
                axis=alt.Axis(format=".0%")
            ),
            y=alt.Y(
                "state:N",
                title=None,
                sort="-x",
                axis=alt.Axis(
                    labels=False,
                    ticks=False
                )
            ),
            color=policy_color,
            stroke=alt.condition(
                alt.datum.state == highlighted_state,
                alt.value("black"),
                alt.value(None)
            ),
            strokeWidth=alt.condition(
                alt.datum.state == highlighted_state,
                alt.value(2),
                alt.value(0)
            ),
            tooltip=[
                "state:N",
                alt.Tooltip(
                    "pass_rate_2025:Q",
                    title="AP Exam Pass Rate",
                    format=".1%"
                )
            ]
        )
        .properties(
            width=350,
            height=180,
            title="AP Exam Pass Rate"
        )
    )


    # ----------------------------------------------
    # EXAMS SCORING 4–5 BAR CHART
    # ----------------------------------------------

    task3_high_score_bar = (
        alt.Chart(task3_data)
        .mark_bar()
        .encode(
            x=alt.X(
                "high_score_rate_2025:Q",
                title="AP Exams Scoring 4–5",
                scale=alt.Scale(domain=[0, 0.60]),
                axis=alt.Axis(format=".0%")
            ),
            y=alt.Y(
                "state:N",
                title=None,
                sort="-x",
                axis=alt.Axis(
                    labels=False,
                    ticks=False
                )
            ),
            color=policy_color,
            stroke=alt.condition(
                alt.datum.state == highlighted_state,
                alt.value("black"),
                alt.value(None)
            ),
            strokeWidth=alt.condition(
                alt.datum.state == highlighted_state,
                alt.value(2),
                alt.value(0)
            ),
            tooltip=[
                "state:N",
                alt.Tooltip(
                    "high_score_rate_2025:Q",
                    title="AP Exams Scoring 4–5",
                    format=".1%"
                )
            ]
        )
        .properties(
            width=350,
            height=180,
            title="AP Exams Scoring 4–5"
        )
    )


    # ----------------------------------------------
    # EXAMS PER EXAMINEE BAR CHART
    # ----------------------------------------------

    task3_exams_bar = (
        alt.Chart(task3_data)
        .mark_bar()
        .encode(
            x=alt.X(
                "exams_per_examinee_2025:Q",
                title="Exams per AP Examinee",
                scale=alt.Scale(
                    domain=[0, 2.2]
                )
            ),
            y=alt.Y(
                "state:N",
                title=None,
                sort="-x",
                axis=alt.Axis(
                    labels=False,
                    ticks=False
                )
            ),
            color=policy_color,
            stroke=alt.condition(
                alt.datum.state == highlighted_state,
                alt.value("black"),
                alt.value(None)
            ),
            strokeWidth=alt.condition(
                alt.datum.state == highlighted_state,
                alt.value(2),
                alt.value(0)
            ),
            tooltip=[
                "state:N",
                alt.Tooltip(
                    "exams_per_examinee_2025:Q",
                    title="Exams per AP Examinee",
                    format=".2f"
                )
            ]
        )
        .properties(
            width=350,
            height=180,
            title="Exams per AP Examinee"
        )
    )


    # ----------------------------------------------
    # COMBINE VISUALIZATIONS
    # ----------------------------------------------

    task3_dashboard = (
        task3_scatter
        &
        (
            task3_participation_bar
            | task3_pass_rate_bar
        )
        &
        (
            task3_high_score_bar
            | task3_exams_bar
        )
    )


    # ----------------------------------------------
    # DISPLAY DASHBOARD
    # ----------------------------------------------

    with task3_output:
        clear_output(wait=True)
        display(task3_dashboard)


# --------------------------------------------------
# UPDATE WHEN A DROPDOWN CHANGES
# --------------------------------------------------

def update_task3(change=None):
    draw_task3(
        policy_widget.value,
        state_widget.value
    )


policy_widget.observe(
    update_task3,
    names="value"
)

state_widget.observe(
    update_task3,
    names="value"
)

# --------------------------------------------------
# PAGE HEADER / INSTRUCTIONS
# --------------------------------------------------

display(
    HTML(
        """
        <div style="
            width: 725px;
            text-align: center;
            margin-bottom: 15px;
        ">
            <h2 style="
                margin-bottom: 4px;
                font-family: 'Arial Black', Arial, sans-serif;
                font-weight: 900;
            ">
                Do State AP Policies Relate to AP Participation and Performance?
            </h2>

            <div style="
                font-size: 13px;
                font-family: Arial, sans-serif;
            ">
                Compare 2016 state AP policies with multiple measures
                of 2025 AP participation and performance.
            </div>
        </div>
        """
    )
)

display(
    HTML(
        """
        <div style="
            width: 725px;
            margin-top: 5px;
            margin-bottom: 10px;
            font-size: 13px;
            font-family: Arial, sans-serif;
        ">
            <b>Explore the data:</b>

            <ul style="
                margin-top: 5px;
                margin-bottom: 5px;
                padding-left: 22px;
            ">
                <li style="margin-bottom: 4px;">
                    Select an <b>AP Policy</b> to compare states that
                    had the policy with states that did not.
                </li>

                <li>
                    Use <b>Highlight State</b> to locate an individual
                    state across all five charts.
                </li>
            </ul>
        </div>
        """
    )
)

# --------------------------------------------------
# DISPLAY DROPDOWNS
# --------------------------------------------------

display(
    widgets.VBox([
        policy_widget,
        state_widget
    ])
)


# --------------------------------------------------
# COLOR KEY
# --------------------------------------------------

display(
    HTML(
        """
        <div style="
            width: 725px;
            margin-top: 8px;
            margin-bottom: 12px;
            font-size: 13px;
        ">
            <span style="color: steelblue; font-size: 18px;">●</span>
            Has selected policy
            &nbsp;&nbsp;&nbsp;&nbsp;

            <span style="color: lightgray; font-size: 18px;">●</span>
            Does not have selected policy
            &nbsp;&nbsp;&nbsp;&nbsp;

            <span style="
                display:inline-block;
                width:11px;
                height:11px;
                border:2px solid black;
                border-radius:50%;
                vertical-align:middle;
            "></span>
            Highlighted state
        </div>
        """
    )
)


# --------------------------------------------------
# DISPLAY VISUALIZATION
# --------------------------------------------------

display(task3_output)

update_task3()

Output()